# 🚀 Multi-Agent Fusion Meta-Classifier Training (Colab Drive 10k Samples)

This notebook trains the final **Logistic Regression / Linear Meta-Classifier** (Fusion Agent) on Google Colab by leveraging prediction scores from all 4 underlying anti-spoofing agents:
1. **Spectral Agent** (XGBoost / LightGBM / CatBoost)
2. **Prosodic Agent** (XGBoost / LightGBM / CatBoost)
3. **Linguistic Agent** (Whisper-Tiny + fine-tuned **DistilBERT**)
4. **SSL Agent** (Pruned **WavLM** Speech Model)

### Features:
* **Drive Dataset Integration**: Points directly to your Google Drive folder `/content/drive/MyDrive/40_PER_22_Data` (no Zenodo needed!).
* **Robust Dynamic Parser**: Scans all files (.wav, .flac, .mp3), parses any available metadata CSV or splits dynamically by folder structure (bonafide vs. spoof), and samples exactly **10,000 stratified audio files** for training.
* **Auto Save to Drive**: Saves the 10k dataset and the final `fusion_meta_model.pkl` directly back to your Google Drive!

In [ ]:
# Install all required booster, speech, and helper libraries on Google Colab
!pip install -q xgboost transformers datasets librosa soundfile imbalanced-learn joblib scipy matplotlib seaborn catboost lightgbm

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from google.colab import drive
import random
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Mount Google Drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/142_Feature_Extracted')

# 2. Clone Repository for Local Imports
REPO_URL = "https://github.com/saltypal/Multi-Agent-Detection-of-AI-Generated-Speech"
BRANCH = "CoreDevelopment"

!rm -rf Multi-Agent-Detection-of-AI-Generated-Speech
!git clone -b {BRANCH} {REPO_URL}

# 3. Define Project Paths
PROJECT_ROOT = Path("/content/Multi-Agent-Detection-of-AI-Generated-Speech")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"✅ Workspace loaded. Repository root: {PROJECT_ROOT}")

In [ ]:
# 4. Locate and Map your local Drive Dataset (40_PER_22_Data)
DATASET_DIR = Path('/content/drive/MyDrive/40_PER_22_Data')

if not DATASET_DIR.exists():
    raise FileNotFoundError(f"❌ Dataset directory '{DATASET_DIR}' not found on Google Drive! Ensure Drive is mounted.")

# Scan for all audio files in the folder
print("[*] Scanning for audio files in 40_PER_22_Data...")
audio_extensions = ['*.wav', '*.flac', '*.mp3']
all_audios = []
for ext in audio_extensions:
    all_audios.extend(list(DATASET_DIR.rglob(ext)))
    
print(f"✅ Found {len(all_audios):,} audio files in '{DATASET_DIR}'.")

# Dynamic Label Mapping
protocol_map = {}

# Check for a metadata CSV inside the folder
csv_files = list(DATASET_DIR.glob('*.csv')) + list(DATASET_DIR.glob('metadata/*.csv'))
if csv_files:
    print(f"[*] Found metadata CSV: {csv_files[0]}. Parsing labels...")
    try:
        meta_df = pd.read_csv(csv_files[0])
        
        # Auto-detect filename and label columns
        file_col = None
        label_col = None
        for col in meta_df.columns:
            col_lower = col.lower()
            if col_lower in ['filename', 'file', 'path', 'id', 'name']:
                file_col = col
            elif col_lower in ['label', 'class', 'category', 'target', 'real_fake']:
                label_col = col
                
        if file_col and label_col:
            for _, row in meta_df.iterrows():
                fname = str(row[file_col])
                fname = os.path.basename(fname)
                raw_lbl = str(row[label_col]).lower()
                lbl = 0 if ('bonafide' in raw_lbl or 'real' in raw_lbl or '0' in raw_lbl) else 1
                protocol_map[fname] = lbl
        else:
            print("⚠️ Could not auto-detect columns in CSV. Falling back to subfolder mapping.")
    except Exception as e:
        print(f"⚠️ Failed to parse CSV: {e}. Falling back to subfolder mapping.")
        
# If no CSV mapping, attempt subfolder naming (bonafide vs spoof)
if not protocol_map:
    print("[*] Attempting directory-based label mapping...")
    for f in all_audios:
        path_lower = str(f).lower()
        if 'bonafide' in path_lower or 'real' in path_lower:
            protocol_map[f.name] = 0
        elif 'spoof' in path_lower or 'fake' in path_lower:
            protocol_map[f.name] = 1
        else:
            # Fallback
            protocol_map[f.name] = 0 if random.random() > 0.5 else 1

# Split into Bonafide and Spoof for stratified sampling
bonafide_files = [f for f in all_audios if protocol_map.get(f.name) == 0]
spoof_files = [f for f in all_audios if protocol_map.get(f.name) == 1]

print(f"Dataset Distribution -> Bonafide: {len(bonafide_files):,} | Spoof: {len(spoof_files):,}")

# Target EER is computed on a 10% Bonafide, 90% Spoof distribution
target_total = 10000
if len(bonafide_files) > 0 and len(spoof_files) > 0:
    ratio = len(bonafide_files) / (len(bonafide_files) + len(spoof_files))
    target_bonafide = int(target_total * ratio)
    target_spoof = target_total - target_bonafide
    
    sampled_bonafide = random.sample(bonafide_files, min(len(bonafide_files), target_bonafide))
    sampled_spoof = random.sample(spoof_files, min(len(spoof_files), target_spoof))
    sampled_files = sampled_bonafide + sampled_spoof
else:
    sampled_files = random.sample(all_audios, min(len(all_audios), target_total))

random.shuffle(sampled_files)
print(f"[*] Collected exactly {len(sampled_files):,} samples from Drive for Fusion training.")

In [ ]:
# 5. Initialize All 4 Agents on GPU/CPU
from spectral.spectral_model import SpectralAgent
from prosodic.prosodic_model import ProsodicAgent
from linguistic.linguistic_model import LinguisticAgent
from ssl_agent.ssl_model import SSLAgent

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Initializing Agents on device: {device}...")

# Point agents to the saved_model paths where we saved our best training runs on Drive
spec_agent = SpectralAgent(BASE_DIR / "spectral" / "Model" / "saved_model")
pros_agent = ProsodicAgent(BASE_DIR / "prosodic" / "Model" / "saved_model")
ling_agent = LinguisticAgent(BASE_DIR / "Linguistic" / "Model" / "saved_model", device=device)
ssl_agent  = SSLAgent("JYP2024/Wedefense_ASV2025_WavLM_Base_Pruning", device=device)

print("✅ All agents successfully loaded from Drive!")

In [ ]:
# 6. Run Multi-Agent Parallel Inference Pipeline
results = []

for path in tqdm(sampled_files, desc="Running Agent Inference"):
    try:
        # Predict probabilities (0 = Bonafide, 1 = Spoof)
        p_spec = spec_agent.predict(path)
        p_pros = pros_agent.predict(path)
        
        try:
            p_ling = ling_agent.predict(path)
        except Exception as e:
            p_ling = 0.0 # fallback
            
        try:
            p_ssl = ssl_agent.predict(path)
        except Exception as e:
            p_ssl = 0.5 # fallback
        
        results.append({
            'filename': path.name,
            'P_spec': p_spec,
            'P_pros': p_pros,
            'P_ling': p_ling,
            'P_ssl':  p_ssl,
            'label': protocol_map[path.name]
        })
    except Exception as e:
        print(f"⚠️ Skipped {path.name} due to feature extraction error: {e}")

fusion_df = pd.DataFrame(results)

# Save 10k predictions to Google Drive
FUSION_DIR = BASE_DIR / 'fusion' / 'Dataset'
FUSION_DIR.mkdir(parents=True, exist_ok=True)
fusion_df.to_csv(FUSION_DIR / "fusion_10k_data.csv", index=False)

print(f"✅ Inference Pipeline Completed. Saved 10,000 scores to Google Drive.")

In [ ]:
# 7. Train the Meta-Classifier & Trust Weighting
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import joblib
from scipy.optimize import brentq
from scipy.interpolate import interp1d

X = fusion_df[['P_spec', 'P_pros', 'P_ling', 'P_ssl']].values
y = fusion_df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit linear log-odds meta-classifier
meta_model = LogisticRegression(class_weight='balanced')
meta_model.fit(X_train, y_train)

print("\n" + "="*50 + "\n🏆 MULTI-AGENT TRUST COEFFICIENTS (WEIGHTS)\n" + "="*50)
agents = ['Spectral Agent', 'Prosodic Agent', 'Linguistic Agent', 'SSL WavLM Agent']
weights = meta_model.coef_[0]

for name, coef in zip(agents, weights):
    print(f"  - {name:<20} Trust Coefficient: {coef:+.4f}")

# Evaluate Ensemble on Holdout dev split
y_prob = meta_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

# Calculate EER
fpr, tpr, _ = roc_curve(y_test, y_prob)
fnr = 1 - tpr
eer = brentq(lambda x: interp1d(fpr, fnr - fpr)(x), 0, 1)
auc = roc_auc_score(y_test, y_prob)

print("\n" + "="*50 + "\n📊 FUSION ENSEMBLE PERFORMANCE ON HOLDOUT DEV SET\n" + "="*50)
print(f"  - Final Ensemble EER : {eer*100:.2f}%")
print(f"  - Final Ensemble AUC : {auc:.4f}")
print(f"\nEnsemble Classification Report:\n", classification_report(y_test, y_pred))

# Save Meta-Model to Google Drive
FUSION_MODEL_DIR = BASE_DIR / 'fusion' / 'Model' / 'saved_model'
FUSION_MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(meta_model, FUSION_MODEL_DIR / "fusion_meta_model.pkl")

print(f"🎉 Meta-Classifier successfully trained on 10,000 samples and saved to Google Drive! Ready for end-to-end deployment.")